In [2]:
import pltkit
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import xarray as xr
from scipy.stats import qmc
import sys, os, glob, re
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shapereader
from matplotlib.ticker import FuncFormatter
sys.path.append(os.path.abspath(".."))
import plotly.graph_objects as go  
import models.Carleton2022.model.mortality_functions as mf
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec

In [ ]:
# Load regional values for insets

wdir = "X:\\user\\liprandicn\\Projects\\mt-comparison\\models\\"
colors_list = ["#C8553D", "#566E3D", "#222E50", "#FEA82F", "#829191"]
temp_type = "heat"
age_group = "oldest"
cause = "All causes"
rt = "IMAGE"
var = "mortality"

models = {
    "hon_file" : [
        0,
        "honda2014/output/ComparisonHonda/mortality_ComparisonHonda_SSP2_ERA5_1980-2023_counterfactual",
        "Honda et al., 2014",
        "H"
        ],
    "sco_file" : [
        1,
        "scovronick2024/output/ComparisonScovronick/mortality_ComparisonScovronick_SSP2_ERA5_1980-2023_counterfactual",
        "Scovronick et al., 2024",
        "S"
        ],
    "car_file" : [
        2,
        "carleton2022/output/ComparisonCarletonCounter/mortality_ComparisonCarletonCounter_SSP2_ERA5_NoAdap_1980-2023_*",
        "Carleton et al., 2022",
        "C"
        ],       
    "bur_file" : [
        3,
        "burkart2022/output/ComparisonBurkart/mortality_ComparisonBurkart_SSP2_ERA5_1980-2023_counterfactual_*",
        "Burkart et al., 2021",
        "B"
        ]
    }

region_vals={}
for region in pltkit.IMAGE_REGIONS:
    region_vals[region] = {"main": {}, "lower": {}, "upper": {}}
    for i,model in enumerate(models):
        m, l, u = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, region, temp_type, cause, age_group, var, None)
        region_vals[region]["main"][model] = m.values
        region_vals[region]["lower"][model] = l.values
        region_vals[region]["upper"][model] = u.values
        
    # region_vals[region]["name"] = regions[region]

In [ ]:
years=range(1980,2024)

rows = []
# Recorremos la estructura anidada
for region, sub_dict in region_vals.items():
    for main_key, files_dict in sub_dict.items():
        for file_name, array_data in files_dict.items():
            for val, year in zip(array_data, years):
                rows.append({
                    'Region': region,
                    'Type': main_key,
                    'File': file_name,
                    'Value': val,
                    "Year":year
                })

# Creamos el DataFrame
df = pd.DataFrame(rows)

df = df.pivot_table(
    index=['Region', 'File', "Type"], 
    columns='Year', 
    values='Value'
).reset_index()


df.to_csv("X:\\user\\liprandicn\\projects\\mt-comparison\\figures\\Paper1\\Appendix_HistoricalMortality_Continents.csv", index=False)

In [ ]:
continents= {
    "Northern America": ["CAN", "USA"],
    "Latin America and the Caribbean": ["MEX", "RCAM", "BRA", "RSAM"],
    "Europe": ["WEU", "CEU", "UKR"],
    "Africa": ["NAF", "WAF", "EAF", "SAF", "RSAF"],
    "Asia": ["TUR", "STAN", "RUS", "ME", "INDIA", "KOR", "CHN", "SEAS", "INDO", "JAP", "RSAS"],
    "Oceania": ["OCE"]
    }
region_to_continent = {
    region: continent 
    for continent, regions in continents.items() 
    for region in regions
}
df['Continent'] = df['Region'].map(region_to_continent)
df = df.groupby(['Continent', 'File', 'Type']).sum(numeric_only=True).reset_index()
continents.keys()


In [ ]:
fig, axs = plt.subplots(2,3, figsize=(10,6), dpi=300)
axs = axs.flatten()

colors_list = ["#C8553D", "#566E3D", "#222E50", "#FEA82F", "#829191"]

labelsize=8
titlesize=11
lettersize=7
years_vlines = [1998, 2003, 2010, 2016, 2019, 2022]

for i,region in enumerate(list(continents.keys())):
    for j, model in enumerate(models):
        main = df[(df['Continent'] == region) & (df['File'] == model) & (df["Type"]=="main")].iloc[:,3:]
        lower = df[(df['Continent'] == region) & (df['File'] == model) & (df["Type"]=="lower")].iloc[:,3:]
        upper = df[(df['Continent'] == region) & (df['File'] == model) & (df["Type"]=="upper")].iloc[:,3:]
        
        axs[i].plot(main.columns, main.values[0], label=models[model][2], linewidth=1, c=colors_list[j], clip_on=False)
        fill = axs[i].fill_between(main.columns.astype(int), lower.values[0], upper.values[0], color=colors_list[j], alpha=0.3)
        fill.set_clip_on(False)

    for year in years_vlines:
        axs[i].axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        
    axs[i].set_title(f"{region}", fontsize=titlesize, y=1.1)
    axs[i].tick_params(axis='both', labelsize=labelsize)
    axs[i].yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:g}k'))
    axs[i].spines["top"].set_visible(False)
    axs[i].spines["right"].set_visible(False)
    
    if (i==0) or (i==3):
        axs[i].set_ylabel("Global excess mortality", fontsize=labelsize)
        
axs[0].set_ylim(-10e3,25e3)
axs[1].set_ylim(-2e3,100e3)
axs[2].set_ylim(-10e3,105e3)
axs[3].set_ylim(-10e3,86e3)
axs[4].set_ylim(-100e3,500e3)
axs[5].set_ylim(-0.5e3,2.1e3)


handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles=handles,
    labels=labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=len(models),
    frameon=False,
    fontsize=labelsize
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()